In [0]:

import sys
from pathlib import Path

print("Python funcionando!")
print("Versão:", sys.version)
print("Diretório atual:", Path.cwd())


In [0]:

from pathlib import Path

# Caminho da pasta com os documentos
DOCS_DIR = Path.cwd() / "docs"

# Listar todos os arquivos PDF
pdfs = sorted(DOCS_DIR.glob("*.pdf"))

print(f"Documentos encontrados: {len(pdfs)}")
for pdf in pdfs:
    tamanho_mb = pdf.stat().st_size / (1024 * 1024)
    print(f"  {pdf.name} ({tamanho_mb:.2f} MB)")

if not pdfs:
    print("Erro: nenhum PDF encontrado em docs/.")


In [0]:

%pip install pypdf
#A biblioteca pypdf permite ler documentos PDF e extrair o texto de suas páginas

In [0]:

import re
from pypdf import PdfReader

def limpar_texto(texto):
    """Normaliza quebras de linha do PDF.
    
    Remove \\n no meio de frases (linhas sem pontuação final),
    preserva parágrafos (\\n\\n) e quebras após . ! ? : ;
    """
    texto = texto.replace("\r\n", "\n")
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    
    linhas = texto.split('\n')
    resultado = []
    buffer = ""
    
    for linha in linhas:
        linha = linha.strip()
        if not linha:
            # Linha vazia = parágrafo
            if buffer:
                resultado.append(buffer)
                buffer = ""
            resultado.append("")
            continue
        
        if buffer:
            # Se o buffer termina com pontuação de fim de frase, fechar
            if buffer[-1] in '.!?;:)]':
                resultado.append(buffer)
                buffer = linha
            else:
                # Senão, juntar com espaço (continua a frase)
                buffer += " " + linha
        else:
            buffer = linha
    
    if buffer:
        resultado.append(buffer)
    
    texto_limpo = '\n'.join(resultado)
    texto_limpo = re.sub(r' {2,}', ' ', texto_limpo)
    return texto_limpo.strip()

# Ler todos os PDFs da pasta docs/
paginas = []

for pdf_path in pdfs:
    try:
        reader = PdfReader(str(pdf_path))
        arquivo = pdf_path.name

        for numero, pagina in enumerate(reader.pages, start=1):
            texto = pagina.extract_text() or ""
            texto = limpar_texto(texto)

            # Filtrar páginas muito curtas (vazias ou <50 chars)
            if len(texto) >= 50:
                paginas.append({
                    "pagina": numero,
                    "texto": texto,
                    "arquivo": arquivo
                })
    except Exception as e:
        print(f"Erro ao ler {pdf_path.name}: {e}")

# Exibir estatísticas
print("Total de páginas com texto:", len(paginas))
print("Caracteres extraídos:", sum(
    len(p["texto"]) for p in paginas
))

print("\nDocumentos processados:")
arquivos_processados = set(p["arquivo"] for p in paginas)
for arq in sorted(arquivos_processados):
    n = sum(1 for p in paginas if p["arquivo"] == arq)
    print(f"  {arq}: {n} páginas")

# Visualizar um exemplo
print("\nEXEMPLO DE TEXTO EXTRAÍDO (após limpeza):\n")
print(paginas[0]["texto"][:1000])


## Etapa 6. Chunking, divisão do documento em trechos

In [0]:

%pip install -q langchain-text-splitters
#Pacote independente do ecossistema LangChain projetado especificamente para dividir textos extensos em partes menores e gerenciáveis (chamadas de chunks).


In [0]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuração inicial do chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " "]
)

# Dividir o texto, preservando páginas e fonte
chunks = []

for pagina in paginas:
    trechos = splitter.split_text(pagina["texto"])
    arquivo = pagina.get("arquivo", "desconhecido.pdf")
    # Prefixo curto para o ID do chunk
    prefixo = arquivo[:25].replace(" ", "_").replace(",", "").replace(".", "")

    for indice, trecho in enumerate(trechos, start=1):
        chunks.append({
            "id": f"{prefixo}-p{pagina['pagina']:03d}-c{indice:03d}",
            "pagina": pagina["pagina"],
            "fonte": arquivo,
            "texto": trecho
        })

# Estatísticas
print("Total de chunks:", len(chunks))
print("Total de páginas processadas:", len(paginas))
print("Tamanho médio dos chunks:",
      round(sum(len(c["texto"]) for c in chunks) / len(chunks)))

# Exemplo
print("\nEXEMPLO DE CHUNK:\n")
print(chunks[10])


## Etapa 7. Gerar embeddings

Agora vamos transformar os textos em representações numéricas para que o RAG consiga encontrar trechos semanticamente relacionados a uma pergunta.
Suporta português, gera vetores de 384 dimensões e pode ser executado localmente, sem precisar de uma API paga.

In [0]:

%pip install -q sentence-transformers
#Transforma frases, parágrafos e documentos inteiros em representações numéricas chamadas de embeddings (vetores)

In [0]:

from sentence_transformers import SentenceTransformer

# Modelo de embeddings multilíngue
NOME_MODELO = "intfloat/multilingual-e5-small"

modelo_embeddings = SentenceTransformer(NOME_MODELO)

# Selecionar três chunks para teste
textos_teste = [
    "passage: " + chunk["texto"]
    for chunk in chunks[:3]
]

# Gerar os vetores
vetores_teste = modelo_embeddings.encode(
    textos_teste,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Modelo:", NOME_MODELO)
print("Chunks processados:", len(vetores_teste))
print("Dimensões:", vetores_teste.shape)

print("\nPrimeiros valores do vetor:")
print(vetores_teste[0][:8])


## Etapa 8. Criar o banco vetorial com ChromaDB

In [0]:

%pip install -q chromadb
# Instalação do ChromaDB
#O ChromaDB permite armazenar embeddings, documentos e metadados, além de executar consultas por similaridade.


In [0]:

import chromadb

# 1. Preparar os textos
textos = [
    "passage: " + chunk["texto"]
    for chunk in chunks
]

# 2. Gerar embeddings dos chunks
vetores = modelo_embeddings.encode(
    textos,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings gerados:", vetores.shape)

# 3. Inicializar o ChromaDB em memória
cliente_chroma = chromadb.Client()

# Remover coleção anterior se existir (evita conflito de dimensões)
try:
    cliente_chroma.delete_collection("poc_educacao_financeira")
except Exception:
    pass

colecao = cliente_chroma.get_or_create_collection(
    name="poc_educacao_financeira",
    embedding_function=None,
    configuration={
        "hnsw": {"space": "cosine"}
    }
)

# 4. Armazenar textos, embeddings e metadados
colecao.upsert(
    ids=[c["id"] for c in chunks],
    embeddings=vetores.tolist(),
    documents=[c["texto"] for c in chunks],
    metadatas=[
        {
            "pagina": c["pagina"],
            "fonte": c["fonte"]
        }
        for c in chunks
    ]
)

# 5. Verificar o banco
print("Documentos indexados:", colecao.count())
print("Banco vetorial pronto!")

# ==========================================
# Função auxiliar de busca com melhorias
# ==========================================
# - Filtro de distância (remove chunks pouco relevantes)
# - Deduplicação por página (evita chunks repetidos da mesma página)
# - Busca mais chunks que o necessário e retorna apenas os K melhores

DISTANCIA_MAX = 0.25  # threshold de distância cosine

def buscar_com_filtro(pergunta, k=8, incluir_documentos=True):
    """Busca no ChromaDB com filtro de distância e deduplicação por página.
    
    Retorna listas filtradas: documentos, metadatas, distancias.
    """
    vetor = modelo_embeddings.encode(
        "query: " + pergunta,
        normalize_embeddings=True
    )
    
    # Sempre incluir documents para reranking com cross-encoder
    resultados = colecao.query(
        query_embeddings=[vetor.tolist()],
        n_results=k * 3,
        include=["documents", "metadatas", "distances"]
    )
    
    docs = resultados.get("documents", [[]])[0]
    metas = resultados["metadatas"][0]
    dists = resultados["distances"][0]
    
    # Filtrar por distância e deduplicar por página
    filtrados_docs = []
    filtrados_metas = []
    filtrados_dists = []
    paginas_vistas = set()
    
    for doc, meta, dist in zip(docs, metas, dists):
        if dist > DISTANCIA_MAX:
            continue
        chave = (meta["pagina"], meta["fonte"])
        if chave in paginas_vistas:
            continue
        paginas_vistas.add(chave)
        filtrados_docs.append(doc)
        filtrados_metas.append(meta)
        filtrados_dists.append(dist)
        if len(filtrados_metas) >= k:
            break
    
    # Reranking com cross-encoder (reordena por relevância real)
    # DESATIVADO: ms-marco-MiniLM-L-6-v2 é treinado em inglês e piora
    # o ranking para português. Reativar com modelo multilíngue no futuro.
    USE_RERANKER = False
    
    if USE_RERANKER and reranker is not None and len(filtrados_docs) > 1:
        pares = [[pergunta, doc] for doc in filtrados_docs]
        scores = reranker.predict(pares)
        ordem = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        filtrados_docs = [filtrados_docs[i] for i in ordem]
        filtrados_metas = [filtrados_metas[i] for i in ordem]
        filtrados_dists = [filtrados_dists[i] for i in ordem]
    
    # Retornar apenas top-k (e documentos se solicitado)
    if not incluir_documentos:
        filtrados_docs = []
    
    return filtrados_docs[:k], filtrados_metas[:k], filtrados_dists[:k]

# ==========================================
# Reranking com Cross-Encoder
# ==========================================
# O cross-encoder reordena os resultados do ChromaDB analisando
# a pergunta e o chunk juntos (attention bidirecional),
# produzindo um ranking mais preciso que o bi-encoder.

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder carregado: cross-encoder/ms-marco-MiniLM-L-6-v2")


## Etapa 9. Implementar a busca semântica

In [0]:

# Pergunta para testar a recuperação
pergunta = (
    "Como uma pessoa pode se preparar financeiramente "
    "para perder sua principal fonte de renda?"
)

# 1. Buscar os 8 chunks mais relevantes (com filtro e dedup)
docs, metadatas, distancias = buscar_com_filtro(pergunta, k=8)

# 3. Exibir os resultados
print("PERGUNTA:", pergunta)
print("\nTRECHOS RECUPERADOS:\n")

for i, (texto, metadata, distancia) in enumerate(
    zip(docs, metadatas, distancias),
    start=1
):
    print(f"========== RESULTADO {i} ==========")
    print(f"Página do PDF: {metadata['pagina']}")
    print(f"Distância: {distancia:.4f}")
    print(f"Fonte: {metadata['fonte']}")
    print(f"Texto:\n{texto[:600]}")
    print()


## Etapa 10. Testar uma pergunta mais específica

In [0]:

# Nova pergunta, mais específica
pergunta_teste = (
    "O que fazer diante da perda de emprego "
    "ou redução da renda familiar? "
    "Qual a importância da reserva de emergência?"
)

# Recuperar os 8 resultados mais próximos (com filtro e dedup)
docs_t, metadatas_t, distancias_t = buscar_com_filtro(pergunta_teste, k=8)

print("PERGUNTA:", pergunta_teste)

for i, (texto, metadata, distancia) in enumerate(
    zip(docs_t, metadatas_t, distancias_t),
    start=1
):
    print(f"\nRESULTADO {i}")
    print(f"Página: {metadata['pagina']}")
    print(f"Distância: {distancia:.4f}")
    print(f"Texto: {texto[:400]}")


## Etapa 11. Escolher o LLM para gerar as respostas

> Modelo escolhido: GPT OSS 20B

## Etapa 12. Conectar o LLM ao notebook

In [0]:

%pip install -q databricks-openai
# A parceria estratégica e a integração nativa entre a Databricks e a OpenAI (anunciada e expandida em acordos de alto impacto) levam os modelos avançados de Inteligência Artificial da OpenAI diretamente para dentro da plataforma de dados e IA da Databricks


In [0]:

from databricks_openai import DatabricksOpenAI

# Autenticação automática no Databricks
client = DatabricksOpenAI()

# Teste inicial do modelo
resposta = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": "Você é um assistente de educação financeira. Responda em português do Brasil, de forma clara e objetiva."
        },
        {
            "role": "user",
            "content": "O que é uma reserva de emergência? Responda em duas frases."
        }
    ],
    max_tokens=500
)

print(resposta.choices[0].message.content)


## Etapa 13. Construir o RAG completo

![image_1794240735706.png](./image_1794240735706.png "image_1794240735706.png")

In [0]:

# ==========================================
# ETAPA 13 - RAG COMPLETO
# ==========================================

# 1. Pergunta do usuário
pergunta_rag = (
    "Como uma pessoa pode se preparar financeiramente "
    "para perder sua principal fonte de renda?"
)

# 2. Buscar os 8 chunks mais relevantes (com filtro e dedup)
docs_rag, metadatas_rag, distancias_rag = buscar_com_filtro(pergunta_rag, k=8)

# 3. Construir o contexto com referências
trechos = []

for i, (texto, metadata) in enumerate(
    zip(docs_rag, metadatas_rag),
    start=1
):
    trechos.append(
        f"[TRECHO {i} | Página {metadata['pagina']} de {metadata['fonte']}]\n"
        f"{texto}"
    )

if not trechos:
    raise ValueError("Nenhum trecho foi recuperado.")

contexto = "\n\n".join(trechos)

# 5. Instruções para o LLM
system_prompt = """
Você é um assistente de educação financeira.

Responda em português do Brasil, utilizando
exclusivamente as informações do contexto fornecido.

Regras:
1. Não invente informações.
2. Não utilize conhecimentos externos.
3. Fundamente as afirmações nos trechos recuperados.
4. Cite as páginas do PDF que sustentam a resposta.
5. Utilize referências como [Página 42 do PDF].
6. Se não houver informações suficientes,
   declare explicitamente essa limitação.
7. Não apresente recomendações como se fossem
   do Banco Central sem evidência no contexto.
8. Se a informação não está no contexto, diga
   "não encontrei essa informação no material".
9. Seja conciso e objetivo. Evite divagações.
"""

# 6. Enviar pergunta e contexto ao LLM
resposta_rag = client.chat.completions.create(
    model="databricks-llama-4-maverick",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto}\n\n"
                f"PERGUNTA:\n{pergunta_rag}"
            )
        }
    ],
    max_tokens=600,
    temperature=0.0
)

# 7. Exibir resposta
print("PERGUNTA:")
print(pergunta_rag)

print("\nRESPOSTA DO RAG:\n")
print(resposta_rag.choices[0].message.content)

# 8. Mostrar as fontes recuperadas
print("\nPÁGINAS RECUPERADAS:")
for metadata in metadatas_rag:
    print(f"- {metadata['fonte']} | Página {metadata['pagina']}")


## Etapa 14. Recuperação com múltiplas consultas

In [0]:
# ANÁLISE DE QUALIDADE DOS PDFs
from pathlib import Path

docs_dir = Path.cwd() / "docs"
pdfs = sorted(docs_dir.glob("*.pdf"))

print("ANÁLISE DE QUALIDADE DOS PDFs")
print("=" * 80)

problemas = []

for pdf_path in pdfs:
    try:
        reader = PdfReader(str(pdf_path))
        n_paginas = len(reader.pages)
        
        paginas_vazias = 0
        paginas_curtas = 0
        texto_total = 0
        quebras_excessivas = 0  # páginas com \n entre palavras
        
        for pagina in reader.pages:
            texto = (pagina.extract_text() or "").strip()
            texto_total += len(texto)
            if not texto:
                paginas_vazias += 1
            elif len(texto) < 100:
                paginas_curtas += 1
            # Detectar quebras excessivas: razão \n / palavras > 0.5
            palavras = texto.split()
            if palavras and texto.count("\n") / len(palavras) > 0.5:
                quebras_excessivas += 1
        
        media_chars = texto_total / n_paginas if n_paginas > 0 else 0
        status = "OK" if media_chars > 500 else ("BAIXA" if media_chars > 100 else "CRÍTICO")
        
        flags = []
        if paginas_vazias: flags.append(f"{paginas_vazias} vazias")
        if paginas_curtas: flags.append(f"{paginas_curtas} curtas")
        if quebras_excessivas: flags.append(f"{quebras_excessivas} quebras excessivas")
        
        print(f"\n{pdf_path.name[:55]}")
        print(f"  Páginas: {n_paginas} | Média chars/pág: {media_chars:.0f} | {status}")
        if flags:
            print(f"  ⚠ {', '.join(flags)}")
            problemas.append((pdf_path.name, flags))
    except Exception as e:
        print(f"\n{pdf_path.name[:55]} → ERRO: {e}")

print(f"\n{'=' * 80}")
print(f"RESUMO: {len(problemas)} PDFs com problemas de qualidade")
for nome, flags in problemas:
    print(f"  - {nome[:50]}: {', '.join(flags)}")

# Verificar arquivo XLS não processado
xls = list(docs_dir.glob("*.xls"))
if xls:
    print(f"\n⚠ Arquivo não-PDF ignorado: {xls[0].name}")

In [0]:
%pip install -q nltk
#NLTK é necessário para as métricas de geração (BLEU e ROUGE) — tokenização em português.

In [0]:
# ============================================================
# ETAPA 14 — AVALIAÇÃO DO RAG
# ============================================================
# Esta seção transforma a documentação anterior em CÓDIGO FUNCIONAL.
# Vamos avaliar o RAG em dois eixos:
#   1. RECUPERAÇÃO  — Recall@K, Precision@K, MRR, NDCG
#   2. GERAÇÃO      — BLEU, ROUGE (vs. referência) + LLM-as-a-Judge
#
# Para cada métrica, o que ela mede e como é calculada:
#   Recall@K     → dos documentos relevantes (ground truth), quantos
#                  aparecem entre os K recuperados?  (exaustidão)
#   Precision@K  → dos K recuperados, quantos são relevantes?   (precisão)
#   MRR          → 1/posição do 1º relevante — premia acertar cedo.
#   NDCG         → soma ponderada por posição; relevantes no topo
#                  valem mais que relevantes no fim.  (ranking)
#   BLEU         → sobreposição de n-gramas entre resposta gerada
#                  e referência (1 a 4-gramas).
#   ROUGE-L      → maior subsequência comum (LCS) entre gerada e ref.
#   LLM-Judge    → o próprio LLM atribui nota 0-5 para:
#                  groundedness (aderência), relevance (relevância),
#                  factual consistency (consistência factual).
# ============================================================

import math
import numpy as np

# ----------------------------------------------------------------
# 14.1  DATASET DE TESTE (ground truth)
# ----------------------------------------------------------------
# Cada item contém:
#   pergunta          → consulta do usuário
#   paginas_relevantes → páginas do PDF que contêm a resposta
#                        (anotadas manualmente — ground truth)
#   resposta_referencia → resposta-modelo escrita a partir do PDF
# ----------------------------------------------------------------

dataset_avaliacao = [
    {
        "pergunta": "Como uma pessoa pode se preparar financeiramente para perder sua principal fonte de renda?",
        "paginas_relevantes": {
            (28, "caderno_cidadania_financeira.pdf"),
            (35, "caderno_cidadania_financeira.pdf"),
            (42, "caderno_cidadania_financeira.pdf"),
            (79, "caderno_cidadania_financeira.pdf"),
            (17, "GUIA planejamento financeiro.pdf"),
            (8, "GUIA planejamento financeiro.pdf"),
            (2, "Orçamento (Susesp).pdf"),
        },
        "resposta_referencia": (
            "Para se preparar, a pessoa deve manter um orçamento superavitário, "
            "separando parte da renda para poupar antes de pagar despesas. "
            "É recomendado automatizar aportes de poupança, diferenciar "
            "necessidades de desejos e formar uma reserva de emergência. "
            "Na perda de emprego, deve-se reduzir despesas proporcionalmente "
            "à queda de renda para evitar endividamento excessivo."
        ),
    },
    {
        "pergunta": "O que são juros compostos e como funcionam?",
        "paginas_relevantes": {
            (32, "caderno_cidadania_financeira.pdf"),
            (33, "caderno_cidadania_financeira.pdf"),
            (34, "caderno_cidadania_financeira.pdf"),
            (84, "caderno_cidadania_financeira.pdf"),
        },
        "resposta_referencia": (
            "Juros compostos são calculados sobre o capital acrescido dos "
            "juros acumulados, ou seja, juros sobre juros. No exemplo do "
            "caderno, R$1.000 a 5% ao mês passam para R$1.050 no primeiro mês, "
            "R$1.102,50 no segundo, e assim por diante. Quanto maior o prazo, "
            "maior o efeito dos juros compostos, beneficiando quem poupa cedo."
        ),
    },
    {
        "pergunta": "Como planejar a aposentadoria do ponto de vista financeiro?",
        "paginas_relevantes": {
            (73, "caderno_cidadania_financeira.pdf"),
            (74, "caderno_cidadania_financeira.pdf"),
            (75, "caderno_cidadania_financeira.pdf"),
            (76, "caderno_cidadania_financeira.pdf"),
            (77, "caderno_cidadania_financeira.pdf"),
            (78, "caderno_cidadania_financeira.pdf"),
            (15, "GUIA planejamento financeiro.pdf"),
            (11, "GUIA planejamento financeiro.pdf"),
            (12, "GUIA planejamento financeiro.pdf"),
        },
        "resposta_referencia": (
            "O planejamento da aposentadoria deve considerar a incerteza do "
            "futuro e o aumento da expectativa de vida. Recomenda-se iniciar "
            "a reserva para aposentadoria assim que se começa a trabalhar, "
            "poupando parte do salário. É importante conhecer os regimes de "
            "previdência obrigatórios e complementares, seus direitos e "
            "deveres, e investir de forma regular para garantir uma renda "
            "estável na terceira idade."
        ),
    },
    {
        "pergunta": "Quais cuidados devo ter ao contratar um seguro?",
        "paginas_relevantes": {
            (69, "caderno_cidadania_financeira.pdf"),
            (71, "caderno_cidadania_financeira.pdf"),
            (72, "caderno_cidadania_financeira.pdf"),
            (73, "caderno_cidadania_financeira.pdf"),
            (77, "caderno_cidadania_financeira.pdf"),
            (78, "caderno_cidadania_financeira.pdf"),
            (10, "GUIA planejamento financeiro.pdf"),
        },
        "resposta_referencia": (
            "Ao contratar um seguro, é importante comparar preços, pois pode "
            "haver grande variação entre seguradoras. Deve-se entender as "
            "coberturas e exclusões, avaliar as necessidades reais, verificar "
            "a idoneidade da seguradora e ler o contrato com atenção antes "
            "de assinar."
        ),
    },
    {
        "pergunta": "Qual a importância de poupar e como começar?",
        "paginas_relevantes": {
            (28, "caderno_cidadania_financeira.pdf"),
            (29, "caderno_cidadania_financeira.pdf"),
            (30, "caderno_cidadania_financeira.pdf"),
            (34, "caderno_cidadania_financeira.pdf"),
            (16, "GUIA planejamento financeiro.pdf"),
            (2, "Orçamento (Susesp).pdf"),
        },
        "resposta_referencia": (
            "Poupar é fundamental para realizar sonhos, prevenir imprevistos "
            "e construir segurança financeira. Para começar, recomenda-se "
            "pagar a si mesmo primeiro, separando parte da renda antes das "
            "despesas, automatizar investimentos em datas predefinidas e "
            "poupar de forma regular, mesmo que com valores pequenos. "
            "Quanto mais cedo começar, maior o efeito dos juros compostos."
        ),
    },
    {
        "pergunta": "O que é crédito e como usá-lo de forma consciente?",
        "paginas_relevantes": {
            (32, "caderno_cidadania_financeira.pdf"),
            (35, "caderno_cidadania_financeira.pdf"),
            (36, "caderno_cidadania_financeira.pdf"),
            (42, "caderno_cidadania_financeira.pdf"),
            (43, "caderno_cidadania_financeira.pdf"),
            (46, "caderno_cidadania_financeira.pdf"),
        },
        "resposta_referencia": (
            "Crédito é uma fonte adicional de recursos obtidos de terceiros "
            "que possibilita antecipar consumo. Para usá-lo de forma "
            "consciente, deve-se planejar dentro do orçamento, usar para "
            "fins essenciais, evitar endividamento excessivo e reduzir "
            "despesas proporcionalmente em caso de perda de renda."
        ),
    },
]

print(f"Dataset de avaliação criado com {len(dataset_avaliacao)} perguntas.")


# ----------------------------------------------------------------
# 14.2  EXECUTAR RECUPERAÇÃO PARA CADA PERGUNTA
# ----------------------------------------------------------------
# Roda o pipeline de busca (embedding + ChromaDB) e guarda as
# páginas recuperadas para cada pergunta.
# ----------------------------------------------------------------

K = 8  # nº de chunks recuperados (igual ao usado no RAG)

resultados_recuperacao = []

for item in dataset_avaliacao:
    pergunta = item["pergunta"]

    # Busca com filtro de distância e deduplicação
    _, metadatas, distancias = buscar_com_filtro(pergunta, k=K, incluir_documentos=False)

    # Páginas recuperadas (na ordem de relevância)
    paginas_recuperadas = [(m["pagina"], m["fonte"]) for m in metadatas]

    resultados_recuperacao.append({
        "pergunta": pergunta,
        "paginas_recuperadas": paginas_recuperadas,
        "distancias": distancias,
        "paginas_relevantes": item["paginas_relevantes"],
    })

print(f"Recuperação executada para {len(resultados_recuperacao)} perguntas (K={K}).\n")
for r in resultados_recuperacao:
    print(f"Pergunta: {r['pergunta'][:60]}...")
    print(f"  Relevantes:  {sorted(r['paginas_relevantes'])}")
    print(f"  Recuperadas: {r['paginas_recuperadas']}")
    print()


# ----------------------------------------------------------------
# 14.3  MÉTRICAS DE RECUPERAÇÃO
# ----------------------------------------------------------------
# Cada função recebe a lista de páginas recuperadas e o conjunto
# de páginas relevantes (ground truth) e devolve um valor entre 0 e 1.
# ----------------------------------------------------------------

def recall_at_k(recuperadas, relevantes, k):
    """Fração dos relevantes que aparecem nos top-k recuperados."""
    top_k = set(recuperadas[:k])
    if not relevantes:
        return 0.0
    return len(top_k & relevantes) / len(relevantes)


def precision_at_k(recuperadas, relevantes, k):
    """Fração dos top-k recuperados que são relevantes."""
    top_k = set(recuperadas[:k])
    if k == 0:
        return 0.0
    return len(top_k & relevantes) / k


def mrr_score(recuperadas, relevantes):
    """Mean Reciprocal Rank — 1/posição do 1º relevante."""
    for i, pag in enumerate(recuperadas, start=1):
        if pag in relevantes:
            return 1.0 / i
    return 0.0


def ndcg_at_k(recuperadas, relevantes, k):
    """Normalized Discounted Cumulative Gain @K.

    Premia relevantes nas primeiras posições: cada posição i
    contribui com 1/log2(i+1). Normaliza pelo DCG ideal.
    """
    # DCG: soma das relevâncias descontadas por posição
    dcg = 0.0
    for i, pag in enumerate(recuperadas[:k], start=1):
        rel = 1.0 if pag in relevantes else 0.0
        dcg += rel / math.log2(i + 1)

    # IDCG: melhor caso possível (todos relevantes nas primeiras posições)
    n_rel = min(len(relevantes), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, n_rel + 1))

    return dcg / idcg if idcg > 0 else 0.0


# --- Calcular todas as métricas ---
linhas_metricas = []

for r in resultados_recuperacao:
    rec = r["paginas_recuperadas"]
    rel = r["paginas_relevantes"]

    recall = recall_at_k(rec, rel, K)
    precision = precision_at_k(rec, rel, K)
    mrr = mrr_score(rec, rel)
    ndcg = ndcg_at_k(rec, rel, K)

    linhas_metricas.append({
        "pergunta": r["pergunta"][:55],
        "recall@8": round(recall, 3),
        "precision@8": round(precision, 3),
        "MRR": round(mrr, 3),
        "NDCG@8": round(ndcg, 3),
    })

# --- Exibir tabela ---
print("=" * 75)
print("  MÉTRICAS DE RECUPERAÇÃO (Retrieval)")
print("=" * 75)
print(f"{'Pergunta':<55} {'Rec@8':>6} {'Prec@8':>6} {'MRR':>6} {'NDCG':>6}")
print("-" * 75)
for linha in linhas_metricas:
    print(f"{linha['pergunta']:<55} {linha['recall@8']:>6} {linha['precision@8']:>6} {linha['MRR']:>6} {linha['NDCG@8']:>6}")
print("-" * 75)

# Médias
med_recall = np.mean([l["recall@8"] for l in linhas_metricas])
med_precision = np.mean([l["precision@8"] for l in linhas_metricas])
med_mrr = np.mean([l["MRR"] for l in linhas_metricas])
med_ndcg = np.mean([l["NDCG@8"] for l in linhas_metricas])
print(f"{'MÉDIA':<55} {med_recall:>6.3f} {med_precision:>6.3f} {med_mrr:>6.3f} {med_ndcg:>6.3f}")
print("=" * 75)


In [0]:
# ============================================================
# 14.4  MÉTRICAS DE GERAÇÃO (BLEU e ROUGE)
# ============================================================
# Agora avaliamos a QUALIDADE DA RESPOSTA gerada pelo LLM.
# Para cada pergunta:
#   1. Recuperamos os chunks (já feito acima)
#   2. Geramos a resposta com o LLM (mesmo pipeline do RAG)
#   3. Comparamos a resposta gerada com a referência usando:
#      - BLEU-4: sobreposição de n-gramas (1 a 4) entre gerada e ref.
#        Valor 0–1; quanto mais próximo de 1, mais similar.
#      - ROUGE-L: maior subsequência comum (LCS) entre as duas.
#        Valor 0–1; mede recall e precision combinados (F-measure).
# ============================================================

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

# Garantir tokenizador do NLTK
try:
    word_tokenize("teste")
except LookupError:
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)

# ----------------------------------------------------------------
# 14.4a  Funções de tokenização e métricas
# ----------------------------------------------------------------

def tokenizar_pt(texto):
    """Tokeniza texto em português (lowercase)."""
    return word_tokenize(texto.lower(), language="portuguese")


def calcular_bleu(referencia, gerada):
    """BLEU-4 com suavização (SmoothingFunction.method1).

    BLEU = brevity_penalty × exp( Σ w_n × log(p_n) )
    onde p_n é a precisão de n-gramas (1 a 4) e w_n = ¼.
    A suavização evita que p_n = 0 (sem nenhum n-grama 4)
    zere o score inteiro.
    """
    ref_tokens = tokenizar_pt(referencia)
    ger_tokens = tokenizar_pt(gerada)
    smoothie = SmoothingFunction().method1
    return sentence_bleu(
        [ref_tokens], ger_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smoothie,
    )


def calcular_rouge_l(referencia, gerada):
    """ROUGE-L: F-measure da maior subsequência comum (LCS).

    LCS = longest common subsequence entre ref e gerada.
    Recall = LCS / |ref|   Precision = LCS / |gerada|
    F = (2 × P × R) / (P + R)    ← versão beta=1 (F1)
    """
    ref_tokens = tokenizar_pt(referencia)
    ger_tokens = tokenizar_pt(gerada)

    # Tabela de programação dinâmica para LCS
    m, n = len(ref_tokens), len(ger_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i - 1] == ger_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs = dp[m][n]
    if lcs == 0:
        return 0.0

    recall = lcs / m
    precision = lcs / n
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


# ----------------------------------------------------------------
# 14.4b  Gerar respostas do RAG para cada pergunta
# ----------------------------------------------------------------
# Reusa o mesmo system_prompt e pipeline da Etapa 13.

system_prompt_av = """
Você é um assistente de educação financeira.
Responda em português do Brasil, utilizando
exclusivamente as informações do contexto fornecido.

Regras:
1. Não invente informações.
2. Não utilize conhecimentos externos.
3. Fundamente as afirmações nos trechos recuperados.
4. Cite as páginas do PDF que sustentam a resposta.
5. Utilize referências como [Página 42 do PDF].
6. Se não houver informações suficientes,
   declare explicitamente essa limitação.
7. Não apresente recomendações como se fossem
   do Banco Central sem evidência no contexto.
8. Se a informação não está no contexto, diga
   "não encontrei essa informação no material".
9. Seja conciso e objetivo. Evite divagações.
"""

print("Gerando respostas do RAG para cada pergunta...\n")

respostas_rag = []

for i, item in enumerate(dataset_avaliacao, start=1):
    pergunta = item["pergunta"]

    # Recuperar chunks com filtro e dedup
    docs_b, metadatas_b, _ = buscar_com_filtro(pergunta, k=K)

    # Montar contexto
    trechos = []
    for j, (texto, metadata) in enumerate(
        zip(docs_b, metadatas_b),
        start=1,
    ):
        trechos.append(
            f"[TRECHO {j} | Página {metadata['pagina']} do PDF]\n{texto}"
        )
    contexto = "\n\n".join(trechos)

    # Gerar resposta (com retry para respostas vazias)
    resposta_texto = ""
    for tentativa in range(3):
        resp = client.chat.completions.create(
            model="databricks-llama-4-maverick",
            messages=[
                {"role": "system", "content": system_prompt_av},
                {
                    "role": "user",
                    "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
                },
            ],
            max_tokens=600,
            temperature=0.0,
        )
        resposta_texto = resp.choices[0].message.content or ""
        if resposta_texto.strip():
            break
        print(f"    (retry {tentativa+1} para Q{i} — resposta vazia)")

    # Calcular BLEU e ROUGE
    bleu = calcular_bleu(item["resposta_referencia"], resposta_texto)
    rouge = calcular_rouge_l(item["resposta_referencia"], resposta_texto)

    respostas_rag.append({
        "pergunta": pergunta,
        "resposta_rag": resposta_texto,
        "resposta_ref": item["resposta_referencia"],
        "bleu": round(bleu, 4),
        "rouge_l": round(rouge, 4),
    })

    print(f"  [{i}/{len(dataset_avaliacao)}] BLEU={bleu:.4f}  ROUGE-L={rouge:.4f}")

# --- Tabela de métricas de geração ---
print("\n" + "=" * 65)
print("  MÉTRICAS DE GERAÇÃO (BLEU-4 e ROUGE-L)")
print("=" * 65)
print(f"{'Pergunta':<45} {'BLEU-4':>8} {'ROUGE-L':>8}")
print("-" * 65)
for r in respostas_rag:
    print(f"{r['pergunta'][:45]:<45} {r['bleu']:>8} {r['rouge_l']:>8}")
print("-" * 65)

med_bleu = np.mean([r["bleu"] for r in respostas_rag])
med_rouge = np.mean([r["rouge_l"] for r in respostas_rag])
print(f"{'MÉDIA':<45} {med_bleu:>8.4f} {med_rouge:>8.4f}")
print("=" * 65)

# Exibir uma resposta de exemplo para inspeção
print("\n--- EXEMPLO DE RESPOSTA GERADA (Pergunta 1) ---")
print(f"Referência: {respostas_rag[0]['resposta_ref'][:200]}...")
print(f"\nRAG: {respostas_rag[0]['resposta_rag'][:300]}...")


In [0]:
# ============================================================
# 14.5  LLM-AS-A-JUDGE
# ============================================================
# Usa o próprio LLM (databricks-gpt-oss-20b) como "juiz" para
# avaliar três dimensões que métricas automáticas não capturam:
#
#   GROUNDEDNESS (Aderência ao contexto)
#     → A resposta se baseia exclusivamente nos trechos recuperados?
#     → Há alucinação (informações fora do contexto)?
#
#   RELEVANCE (Relevância)
#     → A resposta responde diretamente à pergunta do usuário?
#
#   FACTUAL_CONSISTENCY (Consistência factual)
#     → As afirmações da resposta são coerentes com o contexto?
#
# Cada critério recebe nota 0–5:
#   0 = péssimo  |  3 = aceitável  |  5 = excelente
# ============================================================

import re

# Prompt do juiz: pede ao LLM que avalie a resposta dado o contexto
def avaliar_com_llm(pergunta, contexto, resposta):
    """Envia contexto + resposta ao LLM e recebe notas 0-5."""

    prompt_juiz = f"""Você é um avaliador especialista. Analise a resposta gerada
por um sistema RAG com base nos critérios abaixo.

CONTEXTO RECUPERADO:
{contexto[:2000]}

PERGUNTA DO USUÁRIO:
{pergunta}

RESPOSTA DO RAG:
{resposta[:1000]}

Avalie cada critério com uma nota de 0 a 5 (0 = péssimo, 5 = excelente):

1. GROUNDEDNESS: A resposta se baseia exclusivamente no contexto? Há alucinação?
2. RELEVANCE: A resposta responde diretamente à pergunta?
3. FACTUAL_CONSISTENCY: As afirmações são coerentes com o contexto?

Responda EXATAMENTE no formato (apenas os números):
GROUNDEDNESS: <nota>
RELEVANCE: <nota>
FACTUAL_CONSISTENCY: <nota>"""

    # Retry até 5x para respostas vazias do LLM (gpt-oss-20b é instável)
    texto = ""
    for tentativa in range(5):
        resp = client.chat.completions.create(
            model="databricks-llama-4-maverick",
            messages=[
                {"role": "system", "content": "Você é um avaliador imparcial de sistemas de IA."},
                {"role": "user", "content": prompt_juiz},
            ],
            max_tokens=300,
            temperature=0.1,
        )
        texto = (resp.choices[0].message.content or "").strip()
        if texto:
            break
        print(f"    (judge retry {tentativa+1} — resposta vazia)")

    # Extrair notas (parse com clamping 0-5)
    notas = {}
    for linha in texto.split("\n"):
        for criterio in ["GROUNDEDNESS", "RELEVANCE", "FACTUAL_CONSISTENCY"]:
            if criterio in linha.upper():
                # Extrair o primeiro número após o dois-pontos
                match = re.search(r":\s*([\d.]+)", linha)
                if match:
                    try:
                        nota = float(match.group(1))
                        # Se > 5, pode ser decimal sem ponto (ex: 24 → 2.4)
                        if nota > 5:
                            if nota > 10:
                                nota = nota / 10
                            else:
                                nota = nota / 2
                        notas[criterio.lower()] = min(max(int(round(nota)), 0), 5)
                    except ValueError:
                        pass
    return notas


# --- Executar avaliação do juiz para cada resposta ---
print("Executando LLM-as-a-Judge para cada resposta...\n")

avaliacoes_juiz = []

for i, item in enumerate(dataset_avaliacao, start=1):
    pergunta = item["pergunta"]

    # Reconstruir contexto (mesmos chunks da geração, com filtro e dedup)
    docs_j, metadatas_j, _ = buscar_com_filtro(pergunta, k=K)
    trechos = []
    for j, (texto, metadata) in enumerate(
        zip(docs_j, metadatas_j), start=1
    ):
        trechos.append(f"[Página {metadata['pagina']}] {texto}")
    contexto = "\n\n".join(trechos)

    resposta_rag = respostas_rag[i - 1]["resposta_rag"]

    notas = avaliar_com_llm(pergunta, contexto, resposta_rag)

    avaliacoes_juiz.append({
        "pergunta": pergunta[:45],
        "groundedness": notas.get("groundedness", 0),
        "relevance": notas.get("relevance", 0),
        "factual_consistency": notas.get("factual_consistency", 0),
    })

    print(
        f"  [{i}/{len(dataset_avaliacao)}] "
        f"Ground={notas.get('groundedness', '?')}  "
        f"Relev={notas.get('relevance', '?')}  "
        f"Factual={notas.get('factual_consistency', '?')}"
    )

# --- Tabela de resultados do juiz ---
print("\n" + "=" * 75)
print("  LLM-AS-A-JUDGE (notas 0-5)")
print("=" * 75)
print(f"{'Pergunta':<45} {'Ground':>6} {'Relev':>6} {'Factual':>6}")
print("-" * 75)
for a in avaliacoes_juiz:
    print(f"{a['pergunta']:<45} {a['groundedness']:>6} {a['relevance']:>6} {a['factual_consistency']:>6}")
print("-" * 75)

med_ground = np.mean([a["groundedness"] for a in avaliacoes_juiz if a["groundedness"] > 0]) if any(a["groundedness"] > 0 for a in avaliacoes_juiz) else 0
med_relev = np.mean([a["relevance"] for a in avaliacoes_juiz if a["relevance"] > 0]) if any(a["relevance"] > 0 for a in avaliacoes_juiz) else 0
med_factual = np.mean([a["factual_consistency"] for a in avaliacoes_juiz if a["factual_consistency"] > 0]) if any(a["factual_consistency"] > 0 for a in avaliacoes_juiz) else 0
n_avaliadas = sum(1 for a in avaliacoes_juiz if a["groundedness"] > 0)
print(f"\n  (Média calculada sobre {n_avaliadas}/{len(avaliacoes_juiz)} questões avaliadas com sucesso)")
print(f"{'MÉDIA':<45} {med_ground:>6.1f} {med_relev:>6.1f} {med_factual:>6.1f}")
print("=" * 75)


In [0]:
# ============================================================
# 14.6  RESUMO CONSOLIDADO + RECOMENDAÇÕES DE MELHORIA
# ============================================================
# Reúne todas as métricas em uma única tabela e aponta o que
# alterar no pipeline para melhorar cada métrica.
# ============================================================

print("=" * 75)
print("  RESUMO CONSOLIDADO — AVALIAÇÃO DO RAG")
print("=" * 75)
print()

print("RECUPERAÇÃO (Retrieval) — escala 0 a 1:")
print(f"  Recall@8      = {med_recall:.3f}   (exaustidão: % dos relevantes recuperados)")
print(f"  Precision@8   = {med_precision:.3f}   (precisão: % dos recuperados que são relevantes)")
print(f"  MRR           = {med_mrr:.3f}   (posição média do 1º relevante)")
print(f"  NDCG@8        = {med_ndcg:.3f}   (qualidade do ranking)")
print()

print("GERAÇÃO (Generation) — escala 0 a 1:")
print(f"  BLEU-4        = {med_bleu:.4f}  (sobreposição de n-gramas vs. referência)")
print(f"  ROUGE-L       = {med_rouge:.4f}  (subsequência comum vs. referência)")
print()

print("LLM-AS-A-JUDGE — escala 0 a 5:")
print(f"  Groundedness       = {med_ground:.1f}/5  (aderência ao contexto)")
print(f"  Relevance          = {med_relev:.1f}/5  (relevância da resposta)")
print(f"  Factual Consist.   = {med_factual:.1f}/5  (consistência factual)")
print()
print("=" * 75)

# ----------------------------------------------------------------
# RECOMENDAÇÕES — o que alterar no pipeline para melhorar as métricas
# ----------------------------------------------------------------

recomendacoes = [
    {
        "metrica": "Recall@8 (0.597)",
        "problema": "~40% das páginas relevantes não são recuperadas",
        "acao": (
            "Aumentar K de 5 para 8–10 (mais chunks recuperados). "
            "Considerar hybrid search (BM25 + embeddings) para capturar "
            "termos exatos que o embedding semântico pode perder."
        ),
    },
    {
        "metrica": "Precision@8 (0.367)",
        "problema": "Muitos chunks irrelevantes entre os recuperados",
        "acao": (
            "Adicionar um filtro de distância (remover chunks com "
            "distância > 0.15). Considerar reranking com um modelo "
            "cross-encoder (ex.: ms-marco-MiniLM) sobre os top-20."
        ),
    },
    {
        "metrica": "MRR (0.486) e NDCG (0.567)",
        "problema": "Relevantes nem sempre aparecem nas primeiras posições",
        "acao": (
            "Chunks duplicados da mesma página (ex.: [28, 28]) prejudicam "
            "o ranking. Usar "
            "colecao.query com where de página única ou deduplicar "
            "pós-recuperação. Melhorar o chunk_size (600→500) e "
            "overlap (120→100) para trechos mais focados."
        ),
    },
    {
        "metrica": "BLEU-4 e ROUGE-L (baixos)",
        "problema": "Respostas geradas diferem muito da referência",
        "acao": (
            "BLEU/ROUGE baixos NÃO significam necessariamente má "
            "qualidade — a resposta pode ser correta mas com palavras "
            "diferentes da referência. Para melhorar: refinar o "
            "system_prompt pedindo respostas mais concisas e "
            "estruturadas, próximas ao estilo da referência."
        ),
    },
    {
        "metrica": "Groundedness / Factual Consistency",
        "problema": "Risco de alucinação se contexto for fraco",
        "acao": (
            "Reforçar no prompt: "
            "'Se a informação não está no contexto, diga \"não encontrei\"'. "
            "Reduzir max_tokens (800→600) para evitar enrolação. "
            "Usar temperature=0.0 na geração para respostas determinísticas."
        ),
    },
    {
        "metrica": "Modelo de embeddings",
        "problema": "multilingual-e5-small (384 dim) é leve mas menos preciso",
        "acao": (
            "Atualizar para multilingual-e5-base (768 dim) ou "
            "bge-large-en-v1.5 (1024 dim) para melhor qualidade "
            "semântica. Contrapartida: mais memória e tempo de encoding."
        ),
    },
    {
        "metrica": "Chunking",
        "problema": "Chunk_size=800 pode quebrar informações no meio",
        "acao": (
            "Experimentar chunk_size=500 com overlap=100 para trechos "
            "mais coesos. Adicionar separadores específicos "
            "(separators=['\n\n', '\n', '. ', ' ']) no splitter "
            "para respeitar parágrafos e frases."
        ),
    },
]

print("\nRECOMENDAÇÕES DE MELHORIA:")
print("=" * 75)
for rec in recomendacoes:
    print(f"\n• {rec['metrica']}")
    print(f"  Problema: {rec['problema']}")
    print(f"  Ação:     {rec['acao']}")
print("\n" + "=" * 75)
